In [ ]:
"""
iAsync Lead Finder — Google Places API se verified business leads
-------------------------------------------------------------------
Run: python lead_finder.py
Install deps: pip install requests pandas
"""

import requests
import pandas as pd
import time
from datetime import date
FIELD_MASK = (
    "places.displayName,places.formattedAddress,places.internationalPhoneNumber,"
    "places.websiteUri,places.rating,places.userRatingCount,places.businessStatus,"
    "nextPageToken"
)

# ---- CONFIG: apni search yahan customize karo ----
SEARCHES = [
    {"query": "roofing company", "location": "London, UK"},
    {"query": "arborist", "location": "Manchester, UK"},
    {"query": "pest control", "location": "Toronto, Canada"},
]

ONLY_NO_WEBSITE = False  # True karo agar sirf no-website leads chahiye
# ---------------------------------------------------


def text_search(query, location, page_token=None):
    url = "https://places.googleapis.com/v1/places:searchText"
    headers = {
        "Content-Type": "application/json",
        "X-Goog-Api-Key": API_KEY,
        "X-Goog-FieldMask": FIELD_MASK,
    }
    body = {"textQuery": f"{query} in {location}"}
    if page_token:
        body["pageToken"] = page_token

    resp = requests.post(url, headers=headers, json=body)
    if resp.status_code != 200:
        raise RuntimeError(f"Places API error ({resp.status_code}): {resp.text}")
    return resp.json()


def collect_leads(query, location):
    leads = []
    page_token = None

    while True:
        data = text_search(query, location, page_token)
        places = data.get("places", [])

        for p in places:
            status = p.get("businessStatus")
            if status and status != "OPERATIONAL":
                continue

            leads.append({
                "name": p.get("displayName", {}).get("text", ""),
                "address": p.get("formattedAddress", ""),
                "phone": p.get("internationalPhoneNumber", ""),
                "website": p.get("websiteUri", ""),
                "rating": p.get("rating", ""),
                "reviews": p.get("userRatingCount", ""),
                "search_query": query,
                "search_location": location,
            })

        page_token = data.get("nextPageToken")
        if not page_token:
            break
        time.sleep(2)

    return leads


def main():
    if API_KEY == "YOUR_GOOGLE_PLACES_API_KEY":
        print("Pehle API_KEY set karo script ke top mein.")
        return

    all_leads = []

    for s in SEARCHES:
        print(f'Searching: "{s["query"]}" in {s["location"]}...')
        try:
            leads = collect_leads(s["query"], s["location"])
            print(f"  -> {len(leads)} results mile")
            all_leads.extend(leads)
        except Exception as e:
            print(f"  -> Error: {e}")

    df = pd.DataFrame(all_leads)

    if not df.empty:
        df = df.drop_duplicates(subset=["name", "phone"])
        if ONLY_NO_WEBSITE:
            df = df[df["website"] == ""]
        df = df.reset_index(drop=True)

    filename = f"leads_{date.today().isoformat()}.csv"
    df.to_csv(filename, index=False)
    print(f"\nDone. {len(df)} unique leads -> {filename}")


if __name__ == "__main__":
    main()